In [2]:
# %cd "drive/MyDrive/Colab Notebooks/QNLPModelTraining"
import os
import re
import sys
import json
import pickle
import contextlib
import numpy as np
from tqdm import tqdm
from typing import List, Tuple, Dict, Optional, Any

import pathlib
from typing import Union
from pathlib import Path
import random

from datetime import datetime
import socket

from qiskit_aer import AerSimulator
from pytket.extensions.qiskit.backends.aer import AerBackend
# from qiskit.providers.aer import AerSimulator
# from pytket.extensions.qiskit import AerBackend

from lambeq.backend.grammar import Diagram, Id
from lambeq import (
    QuantumTrainer,
    TketModel, Dataset,
    SPSAOptimizer,
    BinaryCrossEntropyLoss,
    AtomicType, IQPAnsatz,
)



In [3]:
def create_backend_config(shots = 1024) -> Dict[str, Any]:
    simulator = AerSimulator(
        method="statevector",
        device="GPU",
        precision="single",
        cuStateVec_enable=True,
    )

    backend = AerBackend(simulation_method="statevector")
    backend._qiskit_backend = simulator

    # print("Available devices:", simulator.available_devices())
    # print("AerSimulator available devices:", AerSimulator().available_devices())
    # print("Backend available devices:", AerBackend()._qiskit_backend.available_devices())

    return {
        "backend": backend,
        "compilation": backend.default_compilation_pass(2),
        "shots": shots,
    }

In [4]:
# from lambeq import (
#     DepCCGParser,
#     # IQPAnsatz,
#     # AtomicType,
#     Rewriter,
#     RemoveCupsRewriter,
#     UnifyCodomainRewriter,
#     SimpleRewriteRule
# )


# ansatz    = IQPAnsatz(
#     {AtomicType.SENTENCE: 1,
#      AtomicType.NOUN:     1, # Galima priskirti 2 qubitus, jei, pvz, treniravimo rezultatai yra prasti
#      }, # AtomicType.PREPOSITIONAL_PHRASE: 0,
#     n_layers=1, n_single_qubit_params=3
# )

# def create_rewriter():
#     # Rule to delete conjunction boxes (“and”, “but”) # just the wire, no box
#     # conj_rule = SimpleRewriteRule(cod=AtomicType.CONJUNCTION, template=Id(AtomicType.CONJUNCTION))

#     rewriter = Rewriter([
#         'determiner',
#         'auxiliary',
#         'connector',
#         'prepositional_phrase',
#     ])
    
#     # rewriter.add_rules(conj_rule)
#     return rewriter

# rewriter = create_rewriter()
# remove_cups = RemoveCupsRewriter()
# unify = UnifyCodomainRewriter(output_type=AtomicType.SENTENCE)

# parser = DepCCGParser(model='elmo', device=0)


In [5]:
# ############### DEBUGGING ##############################

# sentences = [
#     "The company announced a new quantum computing platform.",
#     "The weather was sunny during the event.",
#     "The platform could improve language processing.",
#     "The audience applauded at the end of the presentation.",
# ]
# # sentences = ["Alice likes Bob.","Bob likes Alice.","Alice writes code.","Bob reads books.",]

# labels = np.array([
#     [0, 1],
#     [1, 0],
#     [0, 1],
#     [1, 0],
# ])

# raw_diagrams = parser.sentences2diagrams(
#     sentences,
#     suppress_exceptions=True
# )

# circuits, kept_sentences, kept_labels = [], [], []

# for sentence, diagram, label in zip(sentences, raw_diagrams, labels):
#     if diagram is None:
#         print(f"Skipping failed parse: {sentence}")
#         continue

#     print(sentence)
#     # print("diagram cod:", diagram.cod, " | cod length:", len(diagram.cod))

#     try:
#         diagram = rewriter(diagram)
#         diagram = remove_cups(diagram)
#         diagram = diagram.normal_form()
#         # diagram = diagram.pregroup_normal_form()
#         diagram = unify(diagram)
#         diagram = diagram.normal_form()
        
#         # print("after remove_cups cod:", diagram.cod, " | cod length:", len(diagram.cod))
#         # print()

#         circuit = ansatz(diagram)

#         circuits.append(circuit)
#         kept_sentences.append(sentence)
#         kept_labels.append(label)

#     except Exception as e:
#         print(f"Skipping sentence due to diagram/circuit error: {sentence}")
#         print(type(e).__name__, e)

# kept_labels = np.array(kept_labels)

# for sentence, circuit in zip(kept_sentences, circuits):
#     tk_circuit = circuit.to_tk()
#     # print(sentence)
#     print("qubits:", tk_circuit.n_qubits, "| gates:", tk_circuit.n_gates, "| depth:", tk_circuit.depth())
#     # print()

# # print(f"\nUsable circuits: {len(circuits)}")

# if len(circuits) == 0:
#     raise RuntimeError("No valid circuits were produced.")


# model = TketModel.from_diagrams(
#     circuits,
#     backend_config=backend_config,
# )

# model.initialise_weights()
# outputs = model(circuits)
# # print("\nRaw model outputs:")
# print(outputs)

# ########## DEBUGGING END ###############

In [6]:
from typing import Sequence, Mapping
def get_deep_type(obj):
    if isinstance(obj, list):
        # We look at the unique types inside the list to keep it readable
        inner_types = {get_deep_type(item) for item in obj}
        return f"List[{' | '.join(sorted(inner_types))}]"

    elif isinstance(obj, dict):
        # We summarize the types of all keys and all values
        key_types = {get_deep_type(k) for k in obj.keys()}
        val_types = {get_deep_type(v) for v in obj.values()}
        return f"Dict[{' | '.join(sorted(key_types))}, {' | '.join(sorted(val_types))}]"

    else:
        # Return the class name (e.g., 'Diagram' or 'str')
        return type(obj).__name__

def get_deep_shape(obj, level=0):
    indent = "  " * level
    
    # 1. Atomic types (Strings/Bytes) - Check these first 
    # because they are technically Sequences too!
    if isinstance(obj, (str, bytes)):
        return f"{indent}str"

    # 2. Dictionaries (Mappings)
    elif isinstance(obj, Mapping):
        if not obj:
            return f"{indent}dict(len=0)"

        lines = [f"{indent}dict(len={len(obj)})"]
        for key, value in obj.items():
            # Get child shape and strip only the first line's indentation 
            # so we can prefix it with our key label
            child = get_deep_shape(value, level + 1).lstrip()
            lines.append(f"{indent}  key={repr(key)} -> {child}")
        return "\n".join(lines)

    # 3. Sequences (Lists, Tuples, etc.)
    elif isinstance(obj, Sequence):
        name = type(obj).__name__
        if not obj:
            return f"{indent}{name}(len=0)"

        header = f"{indent}{name}(len={len(obj)})"
        
        # Calculate shapes of all children
        child_shapes = [get_deep_shape(item, level + 1).lstrip() for item in obj]
        unique_shapes = sorted(list(set(child_shapes)))

        if len(unique_shapes) == 1:
            # All items are identical structure
            return f"{header}\n{indent}  [*] -> {unique_shapes[0]}"
        else:
            lines = [header]
            for i, shape in enumerate(child_shapes):
                lines.append(f"{indent}  [{i}] -> {shape}")
            return "\n".join(lines)

    # 4. Base objects (int, float, None, etc.)
    else:
        return f"{indent}{type(obj).__name__}"


In [7]:
EncodedBatch = Dict[str, Any]
EncodedArticle = Dict[str, Any]

def load_encoded_batch(path: Union[str, Path]) -> EncodedBatch:
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Pickle file does not exist: {path}")

    if not path.is_file():
        raise ValueError(f"Path is not a file: {path}")

    with open(path, "rb") as f:
        batch = pickle.load(f)

    required_keys = {"encoded_dataset", "errors", "n_qubits"}
    missing = required_keys - set(batch.keys())

    if missing:
        raise KeyError(f"Missing keys in encoded batch {path}: {missing}")

    if not isinstance(batch["encoded_dataset"], list):
        raise TypeError("'encoded_dataset' must be a list of encoded articles.")

    return batch

def load_and_merge_encoded_batches(num_batches: int, start_idx: int = 0) -> EncodedBatch:
    """
    Load a specified number of encoded batch pickle files from 
    Dataset/Encoded/WikiHow/ and merge them into one batch.
    """

    merged_batch: EncodedBatch = {
        "encoded_dataset": [],
        "errors": [],
        "n_qubits": []
    }

    base_path = pathlib.Path("Dataset/Encoded/WikiHow")
    seen_article_ids = set()
    n_sentences = 0
    n_positive_labels = 0

    for i in range(num_batches):
        start = (start_idx + i) * 100
        end = (start_idx + i + 1) * 100
        filename = f"encoded_WikiHow_{start}_{end}.pkl"
        file_path = base_path / filename

        if not file_path.exists():
            print(f"Warning: {file_path} not found. Stopping merge.")
            break

        batch = load_encoded_batch(file_path)

        # Merge dataset and check for duplicates
        for article in batch["encoded_dataset"]:
            article_id = article.get("article_id")
            if article_id in seen_article_ids:
                raise ValueError(
                    f"Duplicate article_id found while merging batches: {article_id}"
                )

            seen_article_ids.add(article_id)
            merged_batch["encoded_dataset"].append(article)
            n_sentences += len(article["circuits"])
            n_positive_labels += article["labels"].count([0,1])

        # Merge metadata lists
        merged_batch["errors"].extend(batch["errors"])
        merged_batch["n_qubits"].extend(batch["n_qubits"])

    print(f"Loaded: {num_batches} batches | {len(seen_article_ids)} articles | {n_sentences} sentences | {n_positive_labels} positive labels")

    return merged_batch

def label_distribution(flat, name: str = "dataset") -> None:
    labels = flat["labels"]

    total = len(labels)
    positives = sum(1 for label in labels if label == [0, 1])
    negatives = sum(1 for label in labels if label == [1, 0])

    if total == 0:
        print(f"{name}: empty")
        return

    # print(f"  total:     {total}")
    print(f"{name}: positive: {positives} ({positives / total:.4f}) | negative: {negatives} ({negatives / total:.4f})")

def split_encoded_batch_aligned(batch: EncodedBatch, train_ratio: float = 0.70, val_ratio: float = 0.15, seed: int = 42 ) -> Tuple[EncodedBatch, EncodedBatch, EncodedBatch]:
    """
    Split an encoded batch at article level while keeping batch-level metadata aligned.

    Returns:
        train_batch, val_batch, test_batch

    Each returned batch has the same structure:
        {
            "encoded_dataset": List[EncodedArticle],
            "errors": List[List[...]],
            "n_qubits": List[List[int]]
        }
    """

    required_keys = {"encoded_dataset", "n_qubits"}
    missing = required_keys - set(batch.keys())

    if missing:
        raise KeyError(f"Missing keys in batch: {missing}")

    encoded_dataset = batch["encoded_dataset"]
    n_qubits = batch["n_qubits"]

    lengths = {
        "encoded_dataset": len(encoded_dataset),
        "n_qubits": len(n_qubits),
    }

    if len(set(lengths.values())) != 1:
        raise ValueError(
            f"Batch-level lists must have the same length, got: {lengths}"
        )

    if not (0 < train_ratio < 1):
        raise ValueError(f"train_ratio must be between 0 and 1, got {train_ratio}")

    if not (0 <= val_ratio < 1):
        raise ValueError(f"val_ratio must be between 0 and 1, got {val_ratio}")

    if train_ratio + val_ratio >= 1:
        raise ValueError(
            f"train_ratio + val_ratio must be less than 1, got "
            f"{train_ratio + val_ratio}"
        )

    indices = list(range(len(encoded_dataset)))

    total_count = len(indices)

    train_end = int(total_count * train_ratio)
    val_end = train_end + int(total_count * val_ratio)

    train_indices = indices[:train_end]
    val_indices = indices[train_end:val_end]
    test_indices = indices[val_end:]

    def make_split(split_indices: List[int]) -> EncodedBatch:
        return {
            "encoded_dataset": [encoded_dataset[i] for i in split_indices],
            "n_qubits": [n_qubits[i] for i in split_indices],
        }

    train_batch = make_split(train_indices)
    val_batch = make_split(val_indices)
    test_batch = make_split(test_indices)

    print(
        "Split complete: "
        f"Train={len(train_batch['encoded_dataset'])}, "
        f"Val={len(val_batch['encoded_dataset'])}, "
        f"Test={len(test_batch['encoded_dataset'])}"
    )

    return train_batch, val_batch, test_batch

def flatten_encoded_dataset(encoded_dataset: List[EncodedArticle], *, keep_metadata: bool = True, validate_lengths: bool = True) -> Dict[str, Any]:
    """
    Flatten article-level encoded dataset into sentence/circuit-level lists.

    Input:
        encoded_dataset = [
            {
                "article_id": int,
                "circuits": List[Diagram],
                "labels": List[List[int]],
                "n_qubits": List[int],
                "original_text_sentences": List[str],
            },
            ...
        ]

    Output:
        {
            "circuits": List[Diagram],
            "labels": List[List[int]],
            "sentences": List[str],
            "article_ids": List[int], # not needed
            "sentence_ids": List[int], # not needed
            "n_qubits": List[int], # not needed
        }
    """

    flat = {
        "circuits": [],
        "labels": [],
    }

    if keep_metadata:
        flat.update({
            "article_ids": [],
            "sentence_ids": [],
            "n_qubits": [],
            "sentences": [],
        })

    for article_idx, article in enumerate(encoded_dataset):
        required_keys = {
            "article_id",
            "circuits",
            "labels",
            "n_qubits",
            "original_text_sentences",
        }

        missing = required_keys - set(article.keys())
        if missing:
            raise KeyError(
                f"Article at index {article_idx} is missing keys: {missing}"
            )

        article_id = article["article_id"]
        circuits = article["circuits"]
        labels = article["labels"]
        n_qubits = article["n_qubits"]
        sentences = article["original_text_sentences"]

        if validate_lengths:
            lengths = {
                "circuits": len(circuits),
                "labels": len(labels),
                "n_qubits": len(n_qubits),
                "original_text_sentences": len(sentences),
            }

            unique_lengths = set(lengths.values())

            if len(unique_lengths) != 1:
                raise ValueError(
                    f"Length mismatch in article_id={article_id}: {lengths}"
                )

        for sentence_idx, circuit in enumerate(circuits):
            flat["circuits"].append(circuit)
            flat["labels"].append(labels[sentence_idx])

            if keep_metadata:
                flat["article_ids"].append(article_id)
                flat["sentence_ids"].append(sentence_idx)
                flat["n_qubits"].append(n_qubits[sentence_idx])
                flat["sentences"].append(sentences[sentence_idx])

    return flat



In [8]:
POSITIVE_INDEX = 1

def _to_classes(y_hat, y, positive_index: int = POSITIVE_INDEX):
    """
    Convert model outputs and one-hot labels to class indices.

    y_hat:
        Model output, usually shape (batch_size, 2)

    y:
        One-hot labels, shape (batch_size, 2)

    Returns:
        y_pred, y_true
    """

    y_hat = np.asarray(y_hat)
    y = np.asarray(y)

    if y_hat.ndim == 1:
        y_hat = y_hat.reshape(1, -1)

    if y.ndim == 1:
        y = y.reshape(1, -1)

    if y_hat.shape[1] != 2:
        raise ValueError(f"Expected y_hat shape (n, 2), got {y_hat.shape}")

    if y.shape[1] != 2:
        raise ValueError(f"Expected y shape (n, 2), got {y.shape}")

    y_pred = np.argmax(y_hat, axis=1)
    y_true = np.argmax(y, axis=1)

    return y_pred, y_true

def accuracy(y_hat, y) -> float:
    y_pred, y_true = _to_classes(y_hat, y)

    return float(np.mean(y_pred == y_true))

def balanced_accuracy(y_hat, y) -> float:
    y_pred, y_true = _to_classes(y_hat, y)

    positive_label = POSITIVE_INDEX
    negative_label = 1 - POSITIVE_INDEX

    tp = np.sum((y_true == positive_label) & (y_pred == positive_label))
    tn = np.sum((y_true == negative_label) & (y_pred == negative_label))

    fn = np.sum((y_true == positive_label) & (y_pred == negative_label))
    fp = np.sum((y_true == negative_label) & (y_pred == positive_label))

    recall_pos = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    recall_neg = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    return float((recall_pos + recall_neg) / 2)

def precision(y_hat, y) -> float:
    """
    Positive-class precision.

    Of all sentences predicted as selected, how many were truly selected?
    """

    y_pred, y_true = _to_classes(y_hat, y)

    positive_label = POSITIVE_INDEX

    tp = np.sum((y_true == positive_label) & (y_pred == positive_label))
    fp = np.sum((y_true != positive_label) & (y_pred == positive_label))

    return float(tp / (tp + fp)) if (tp + fp) > 0 else 0.0

def recall(y_hat, y) -> float:
    """
    Positive-class recall.

    Of all truly selected sentences, how many did the model find?
    """

    y_pred, y_true = _to_classes(y_hat, y)

    positive_label = POSITIVE_INDEX

    tp = np.sum((y_true == positive_label) & (y_pred == positive_label))
    fn = np.sum((y_true == positive_label) & (y_pred != positive_label))

    return float(tp / (tp + fn)) if (tp + fn) > 0 else 0.0

def f1(y_hat, y) -> float:
    """
    Positive-class F1 score.
    """

    p = precision(y_hat, y)
    r = recall(y_hat, y)

    return float(2 * p * r / (p + r)) if (p + r) > 0 else 0.0

def validate_ds_data(train_circuits, train_labels, val_circuits, val_labels, ) -> None:
    if len(train_circuits) != len(train_labels):
        raise ValueError(
            f"Train circuits/labels length mismatch: "
            f"{len(train_circuits)} circuits vs {len(train_labels)} labels"
        )

    if len(val_circuits) != len(val_labels):
        raise ValueError(
            f"Validation circuits/labels length mismatch: "
            f"{len(val_circuits)} circuits vs {len(val_labels)} labels"
        )

    if len(train_circuits) == 0:
        raise ValueError("Training set is empty.")

    if len(val_circuits) == 0:
        raise ValueError("Validation set is empty.")

    for i, label in enumerate(train_labels):
        if len(label) != 2:
            raise ValueError(
                f"Expected train label at index {i} to have length 2, got {label}"
            )

    for i, label in enumerate(val_labels):
        if len(label) != 2:
            raise ValueError(
                f"Expected validation label at index {i} to have length 2, got {label}"
            )

def train_quantum_model(
    train_flat: Dict[str, Any],
    val_flat: Dict[str, Any],
    test_flat: Dict[str, Any],
    # construct_flat,
    backend_config,
    # model = None,
    batch_size: int = 8,
    epochs: int = 5,
    learning_rate: float = 0.05,
    seed: int = 42,
    optimizer_hyperparams: Optional[Dict[str, float]] = None,
    evaluate_on_train: bool = True,
    verbose_level: str = "text",
    log_dir: Path | str | None = None,
    from_checkpoint=True,
) -> Tuple[TketModel, QuantumTrainer]:
    """
    Returns:
        model, trainer

    This function intentionally does NOT include:
        - staged training
        - resume logic
        - test evaluation
        - custom train/validation schedules
    """

    train_circuits = train_flat["circuits"]
    train_labels = train_flat["labels"]
    val_circuits = val_flat["circuits"]
    val_labels = val_flat["labels"]
    test_circuits = test_flat["circuits"]
    # test_labels = test_flat["labels"]
    validate_ds_data(train_circuits, train_labels, val_circuits, val_labels)

    bce = BinaryCrossEntropyLoss()
    if optimizer_hyperparams is None:
        optimizer_hyperparams = {
            "a": learning_rate, # main learning-rate scale
            "c": 0.06, # perturbation size for gradient approximation
            "A": 10, # stability/delay parameter
        }

    eval_metrics = {
        "acc": accuracy,
        "balanced_acc": balanced_accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }
    
    if from_checkpoint:
        if log_dir is None:
            raise ValueError("When from_checkpoint=True, you must provide log_dir.")
        log_dir = Path(log_dir)

        checkpoint_path = log_dir / "model.lt"

        if not checkpoint_path.exists():
            raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

        model = TketModel(backend_config=backend_config)

    else:
        # all_circuits = construct_flat["circuits"]
        all_circuits = train_circuits + val_circuits + test_circuits
        model = TketModel.from_diagrams(
            all_circuits,
            backend_config=backend_config,
        )

    trainer = QuantumTrainer(
        model=model,
        loss_function=bce,
        epochs=epochs,
        # eval_interval=eval_interval,
        optimizer=SPSAOptimizer,
        optim_hyperparams=optimizer_hyperparams,
        evaluate_functions=eval_metrics,
        evaluate_on_train=evaluate_on_train,
        log_dir=log_dir,
        from_checkpoint=from_checkpoint,
        verbose=verbose_level,
        seed=seed,
    )

    train_dataset = Dataset(train_circuits, train_labels,
        batch_size=batch_size, shuffle=True,
    )

    val_dataset = Dataset(val_circuits, val_labels,
        batch_size=batch_size, shuffle=False,
    )

    trainer.fit(train_dataset, val_dataset,
        eval_interval=1, log_interval=1,
    )

    return model, trainer


In [18]:
batch = load_and_merge_encoded_batches(10)
train_data, val_data, test_data = split_encoded_batch_aligned(batch) #, train_ratio=0.95, val_ratio=0.03)

Loaded: 10 batches | 999 articles | 6219 sentences | 1624 positive labels
Split complete: Train=699, Val=149, Test=151


Loaded: 70 batches | 6992 articles | 42601 sentences | 11137 positive labels\
Split complete: Train=5244, Val=1048, Test=700

Loaded: 25 batches | 2495 articles | 15382 sentences | 3975 positive labels\
Split complete: Train=1871, Val=374, Test=250

Loaded: 20 batches | 1995 articles | 12248 sentences | 3175 positive labels\
Split complete: Train=1496, Val=299, Test=200

Loaded: 5 batches | 500 articles | 3075 sentences | 810 positive labels\
Split complete: Train=375, Val=75, Test=50

Loaded: 1 batches | 100 articles | 598 sentences | 161 positive labels\
Split complete: Train=70, Val=15, Test=15

In [19]:
train_flat = flatten_encoded_dataset(train_data["encoded_dataset"])
val_flat = flatten_encoded_dataset(val_data["encoded_dataset"])
test_flat = flatten_encoded_dataset(test_data["encoded_dataset"])
# label_distribution(train_flat, "train")
# label_distribution(val_flat, "val")
# label_distribution(test_flat, "test")


In [12]:
backend_config = create_backend_config(shots=2048)

/home/green/QNLPModelTraining/qnlp_3_10/lib/python3.10/site-packages/pytket/extensions/qiskit/backends/aer.py:129: UserWarning: More than one backend with name 'aer_simulator' is available. Picking one.
  warnings.warn(


In [ ]:
model, trainer = train_quantum_model(
    ten_flat, val_flat, ten_flat, backend_config,
     batch_size=256, epochs=25, learning_rate=0.05,
    # verbose_level="progress",
    log_dir="runs/Main", from_checkpoint=False
)

In [ ]:
# shots: 1024 | load bach size: 1 | bach size in model: 64 | epochs: 5
# Epoch 1:  train/loss: 1.8997   valid/loss: 1.7935   train/time: 2m21s   valid/time: 7.25s   train/acc: 0.6688   valid/acc: 0.6455
# Epoch 2:  train/loss: 2.3409   valid/loss: 1.6778   train/time: 2m20s   valid/time: 7.04s   train/acc: 0.6430   valid/acc: 0.7000
# Epoch 3:  train/loss: 1.9390   valid/loss: 2.0033   train/time: 2m23s   valid/time: 7.61s   train/acc: 0.6538   valid/acc: 0.6636
# Epoch 4:  train/loss: 1.2208   valid/loss: 2.7237   train/time: 2m26s   valid/time: 6.94s   train/acc: 0.6516   valid/acc: 0.6545
# Epoch 5:  train/loss: 1.9812   valid/loss: 1.9412   train/time: 2m20s   valid/time: 6.70s   train/acc: 0.6710   valid/acc: 0.6000
# train/time: 11m49s   train/time_per_epoch: 2m22s   train/time_per_step: 17.73s   valid/time: 1m25s   valid/time_per_eval: 8.51s 


# shots: 2048 | load bach size: 5 | bach size in model: 32 | epochs: 2
# Epoch 1:  train/loss: 1.8154   valid/loss: 2.1592   train/time: 11m40s   valid/time: 3.23s   train/acc: 0.6333   valid/acc: 0.6581
# Epoch 2:  train/loss: 0.6098   valid/loss: 2.0423   train/time: 11m43s   valid/time: 3.42s   train/acc: 0.6415   valid/acc: 0.6731
# train/time: 23m24s   train/time_per_epoch: 11m42s   train/time_per_step: 9.61s   valid/time: 2m21s   valid/time_per_eval: 4.69s


# shots: 2048 | load bach size: 1 | bach size in model: 256 | epochs: 10 | learning_rate=0.05
# train/time: 22m20s   train/time_per_epoch: 2m14s   train/time_per_step: 1m7s   valid/time: 1m56s   valid/time_per_eval: 11.60s



In [ ]:
import matplotlib.pyplot as plt
def get_metric(results, name):
    """
    Supports both possible structures:
    1. {"acc": [0.1, 0.2, ...]}
    2. [{"acc": 0.1}, {"acc": 0.2}, ...]
    """
    if results is None:
        return []

    # Case 1: dict of metric lists
    if isinstance(results, dict):
        values = results.get(name, [])
        return list(values)

    # Case 2: list of dictionaries
    if isinstance(results, list):
        values = []
        for item in results:
            if isinstance(item, dict) and name in item:
                values.append(item[name])
        return values

    return []


def plot_loss(trainer):
    train_loss = list(trainer.train_epoch_costs)
    val_loss = list(trainer.val_costs)

    epochs_train = range(1, len(train_loss) + 1)
    epochs_val = range(1, len(val_loss) + 1)

    plt.figure(figsize=(9, 5))
    plt.plot(epochs_train, train_loss, marker="o", label="Train loss")
    plt.plot(epochs_val, val_loss, marker="o", label="Validation loss")

    plt.title("Training and Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Binary Cross-Entropy Loss")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


def plot_accuracy_metrics(trainer):
    train_acc = get_metric(trainer.train_eval_results, "acc")
    val_acc = get_metric(trainer.val_eval_results, "acc")

    train_bal_acc = get_metric(trainer.train_eval_results, "balanced_acc")
    val_bal_acc = get_metric(trainer.val_eval_results, "balanced_acc")

    plt.figure(figsize=(9, 9))

    if train_acc:
        plt.plot(range(1, len(train_acc) + 1), train_acc, marker="o", label="Train accuracy")
    if val_acc:
        plt.plot(range(1, len(val_acc) + 1), val_acc, marker="o", label="Validation accuracy")

    if train_bal_acc:
        plt.plot(range(1, len(train_bal_acc) + 1), train_bal_acc, marker="o", label="Train balanced accuracy")
    if val_bal_acc:
        plt.plot(range(1, len(val_bal_acc) + 1), val_bal_acc, marker="o", label="Validation balanced accuracy")

    plt.title("Accuracy and Balanced Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    # plt.ylim(0.4, 1.0)
    plt.ylim(0.5, 0.8)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


def plot_positive_class_metrics(trainer):
    metric_names = [
        "precision",
        "recall",
        "f1",
    ]

    display_names = {
        "precision_positive": "Precision positive",
        "recall_positive": "Recall positive",
        "f1_positive": "F1 positive",
    }

    plt.figure(figsize=(10, 5))

    for metric_name in metric_names:
        train_values = get_metric(trainer.train_eval_results, metric_name)
        val_values = get_metric(trainer.val_eval_results, metric_name)

        if train_values:
            plt.plot(
                range(1, len(train_values) + 1),
                train_values,
                marker="o",
                label=f"Train {display_names[metric_name]}"
            )

        if val_values:
            plt.plot(
                range(1, len(val_values) + 1),
                val_values,
                marker="o",
                label=f"Validation {display_names[metric_name]}"
            )

    plt.title("Positive-Class Precision, Recall, and F1")
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.ylim(0.0, 1.0)
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_loss(trainer)

In [ ]:
plot_accuracy_metrics(trainer)

In [ ]:
plot_positive_class_metrics(trainer)

In [ ]:
b = load_and_merge_encoded_batches(5)
# td, vd, tsd = split_encoded_batch_aligned(b, train_ratio=0.7, val_ratio=0.15)

# tf = flatten_encoded_dataset(td["encoded_dataset"])
# vf = flatten_encoded_dataset(vd["encoded_dataset"])
# tsf = flatten_encoded_dataset(tsd["encoded_dataset"])
bf = flatten_encoded_dataset(b["encoded_dataset"])

# label_distribution(tf, "train")
# label_distribution(vf, "val")
# label_distribution(tsf, "test")
label_distribution(bf, "batch")


In [33]:
def evaluate_quantum_predictions(
    model,
    flat_dataset: Dict[str, Any],
    *,
    batch_size: int = 16,
    positive_index: int = 1,
    threshold: Optional[float] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Evaluate a trained lambeq TketModel on a flattened dataset.

    Expected flat_dataset:
        {
            "circuits": List[Diagram],
            "labels": List[List[int]],
            optional:
                "sentences": List[str],
                "article_ids": List[int],
                "sentence_ids": List[int],
                "n_qubits": List[int],
        }

    Label convention:
        [1, 0] = negative / not selected
        [0, 1] = positive / selected

    Prediction convention:
        If threshold is None:
            predicted class = argmax(model_output)

        If threshold is given:
            predicted positive if model_output[:, positive_index] >= threshold

    Returns:
        Dictionary containing predictions, probabilities, and classification metrics.
    """

    circuits = flat_dataset["circuits"]
    labels = flat_dataset["labels"]

    if len(circuits) != len(labels):
        raise ValueError(
            f"Circuits/labels length mismatch: "
            f"{len(circuits)} circuits vs {len(labels)} labels"
        )

    if len(circuits) == 0:
        raise ValueError("Cannot evaluate an empty dataset.")

    y_true_onehot = np.asarray(labels)

    if y_true_onehot.ndim != 2 or y_true_onehot.shape[1] != 2:
        raise ValueError(
            f"Expected labels to have shape (n_samples, 2), "
            f"got {y_true_onehot.shape}"
        )

    y_true = np.argmax(y_true_onehot, axis=1)

    prediction_batches = []

    for start in range(0, len(circuits), batch_size):
        end = start + batch_size
        batch_circuits = circuits[start:end]

        # TketModel.forward(...) calls get_diagram_output(...)
        # and returns an ndarray of model predictions.
        batch_predictions = model.forward(batch_circuits)

        prediction_batches.append(np.asarray(batch_predictions))

    y_prob = np.vstack(prediction_batches)

    if y_prob.ndim != 2 or y_prob.shape[1] != 2:
        raise ValueError(
            f"Expected model predictions to have shape (n_samples, 2), "
            f"got {y_prob.shape}"
        )

    if threshold is None:
        y_pred = np.argmax(y_prob, axis=1)
    else:
        y_pred = (y_prob[:, positive_index] >= threshold).astype(int)

    positive_label = positive_index
    negative_label = 1 - positive_index

    tp = int(np.sum((y_true == positive_label) & (y_pred == positive_label)))
    tn = int(np.sum((y_true == negative_label) & (y_pred == negative_label)))
    fp = int(np.sum((y_true == negative_label) & (y_pred == positive_label)))
    fn = int(np.sum((y_true == positive_label) & (y_pred == negative_label)))

    total = len(y_true)

    accuracy = (tp + tn) / total if total > 0 else 0.0

    precision_pos = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall_pos = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_pos = (
        2 * precision_pos * recall_pos / (precision_pos + recall_pos)
        if (precision_pos + recall_pos) > 0
        else 0.0
    )

    precision_neg = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    recall_neg = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1_neg = (
        2 * precision_neg * recall_neg / (precision_neg + recall_neg)
        if (precision_neg + recall_neg) > 0
        else 0.0
    )

    positive_count = int(np.sum(y_true == positive_label))
    negative_count = int(np.sum(y_true == negative_label))

    predicted_positive_count = int(np.sum(y_pred == positive_label))
    predicted_negative_count = int(np.sum(y_pred == negative_label))

    majority_baseline_acc = max(positive_count, negative_count) / total

    balanced_accuracy = (recall_pos + recall_neg) / 2

    eps = 1e-12
    clipped_probs = np.clip(y_prob, eps, 1.0 - eps)

    binary_cross_entropy = -np.mean(
        np.sum(y_true_onehot * np.log(clipped_probs), axis=1)
    )

    results = {
        "total": total,

        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "majority_baseline_acc": majority_baseline_acc,
        "binary_cross_entropy": binary_cross_entropy,

        "positive_count": positive_count,
        "negative_count": negative_count,
        "positive_ratio": positive_count / total,
        "negative_ratio": negative_count / total,

        "predicted_positive_count": predicted_positive_count,
        "predicted_negative_count": predicted_negative_count,
        "predicted_positive_ratio": predicted_positive_count / total,
        "predicted_negative_ratio": predicted_negative_count / total,

        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,

        "precision_pos": precision_pos,
        "recall_pos": recall_pos,
        "f1_pos": f1_pos,

        "precision_neg": precision_neg,
        "recall_neg": recall_neg,
        "f1_neg": f1_neg,

        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
    }

    if verbose:
        print("Evaluation results")
        print("------------------")
        print(f"Samples:                  {total}")
        print(f"Accuracy:                 {accuracy:.4f}")
        print(f"Balanced accuracy:        {balanced_accuracy:.4f}")
        print(f"Majority baseline acc:    {majority_baseline_acc:.4f}")
        print(f"Binary cross-entropy:     {binary_cross_entropy:.4f}")
        print()
        print(f"True positives:           {positive_count} ({positive_count / total:.4f})")
        print(f"True negatives:           {negative_count} ({negative_count / total:.4f})")
        print(f"Predicted positives:      {predicted_positive_count} ({predicted_positive_count / total:.4f})")
        print(f"Predicted negatives:      {predicted_negative_count} ({predicted_negative_count / total:.4f})")
        print()
        print("Confusion matrix")
        print(f"TP: {tp} | FP: {fp}")
        print(f"FN: {fn} | TN: {tn}")
        print()
        print("Positive class metrics")
        print(f"Precision:                {precision_pos:.4f}")
        print(f"Recall:                   {recall_pos:.4f}")
        print(f"F1:                       {f1_pos:.4f}")
        print()
        print("Negative class metrics")
        print(f"Precision:                {precision_neg:.4f}")
        print(f"Recall:                   {recall_neg:.4f}")
        print(f"F1:                       {f1_neg:.4f}")

    return results

def load_model(dataset_flat, backend_config, path: Path):
    model = TketModel.from_diagrams(
        dataset_flat["circuits"],
        backend_config=backend_config,
    )

    model.load(path)
    # new_model.initialise_weights()
    # old_model = load_model(save_path)
    # new_model = transfer_matching_weights(old_model, new_model)
    return model


In [ ]:
model_lt_path = Path("runs/24h_25_dataBatches/best_model.lt")
backend_config = create_backend_config(shots=2048)

loaded_model = load_model(bf, backend_config, model_lt_path)

In [ ]:
results = evaluate_quantum_predictions(loaded_model, bf, batch_size=64)

In [ ]:
from lambeq.training.checkpoint import Checkpoint

log_dir = "runs/24h_25_data_batches"

ckpt = Checkpoint.from_file(f"{log_dir}/model.lt")
# or:
# ckpt = Checkpoint.from_file(f"{log_dir}/best_model.lt")

print(ckpt.entries.keys())

print("Epoch:", ckpt["epoch"])
print("Step:", ckpt["step"])

print("Train batch losses:", ckpt["train_costs"][:3], len(ckpt["train_costs"]))
print("Train epoch losses:", ckpt["train_epoch_costs"][:3], len(ckpt["train_epoch_costs"]))
print("Validation losses:", ckpt["val_costs"][:3], len(ckpt["val_costs"]))

print("Train metrics:", ckpt["train_eval_results"]["acc"][:3], len(ckpt["train_eval_results"]["acc"]))
print("Validation metrics:", ckpt["val_eval_results"]["acc"][:3], len(ckpt["val_eval_results"]["acc"]))

##### debbuging

In [ ]:
def validate_circuits(model, ds_flat):
    preds = []
    n_errors = 0
    # try:
    #     preds = model.forward(ds_flat["circuits"], )
    # except Exception as e:
    #     print(f" | {e}")

    for i, circuit in enumerate(tqdm(ds_flat["circuits"])):
        try:
            preds.append(model.forward([circuit]))
        except Exception as e:
            print(f"{i} | {e}")
            n_errors += 1
    print("invalid circuits:", n_errors)
    return preds

def transfer_matching_weights(old_model, new_model):
    """
    Copies weights from old_model to new_model when symbols match.
    New symbols stay randomly initialized.
    """

    old_symbols = list(old_model.symbols)
    new_symbols = list(new_model.symbols)

    old_weights = np.array(old_model.weights)
    new_weights = np.array(new_model.weights)

    old_weight_by_symbol = {
        symbol: weight
        for symbol, weight in zip(old_symbols, old_weights)
    }

    copied = 0
    missing = 0

    for i, symbol in enumerate(new_symbols):
        if symbol in old_weight_by_symbol:
            new_weights[i] = old_weight_by_symbol[symbol]
            copied += 1
        else:
            missing += 1

    new_model.weights = new_weights

    print(f"Copied weights: {copied}")
    print(f"New/random weights: {missing}")

    return new_model


In [ ]:
preds_temp = validate_circuits(model, test_flat)

for pred in preds:
    print(pred)

In [ ]:
b = load_and_merge_encoded_batches(5, 1)
td, vd, tsd = split_encoded_batch_aligned(b, train_ratio=0.8, val_ratio=0.15)

tf = flatten_encoded_dataset(td["encoded_dataset"])
vf = flatten_encoded_dataset(vd["encoded_dataset"])
tsf = flatten_encoded_dataset(tsd["encoded_dataset"])
bf = flatten_encoded_dataset(b["encoded_dataset"])

label_distribution(tf, "train")
label_distribution(vf, "val")
label_distribution(tsf, "test")

In [ ]:
p = validate_circuits(new_model, bf)
print("valid circuits:", len(p))

In [ ]:
####################################################
####################################################
####################################################
####################################################

In [ ]:
def set_stage_configs(shots, epochs, use_stage_configs: bool = False):
    if use_stage_configs:
        stage_configs = []
        for s, a in [(2048, 0.1), (4096, 0.05), (8192, 0.02)]:
            stage_configs.append({
                'epochs': epochs,
                'shots': s,
                'optimizer_hparams': {
                    'a': a,
                    'c': 0.06 if s<=4096 else 0.02,
                    'A': 0.2 * epochs
                }
            })
    else:
        # Single “stage” using default parameters
        stage_configs = [{
            'epochs': epochs,
            'shots': shots,
            'optimizer_hparams': { 'a': 0.1, 'c': 0.06, 'A': 0.2 * epochs }
        }]

    return stage_configs

In [ ]:
def train_quantum_summarizer(
    encoded_train: List[Dict],
    encoded_val:   List[Dict],
    encoded_test:  List[Dict],
    batch_size: int = 5,
    epochs:     int = 10,
    shots:      int = 8192, # 4096
    seed:       int = 42,
    checkpoint_dir:    str = 'saves/model_checkpoints',
    use_stage_configs: bool = True,
    stage_configs:     List[Dict] = None
) -> Tuple[TketModel, QuantumTrainer, Dict[str, float]]:

    os.makedirs(checkpoint_dir, exist_ok=True)

    train_circuits, train_labels = flatten(encoded_train)
    val_circuits,   val_labels   = flatten(encoded_val)
    test_circuits,  test_labels  = flatten(encoded_test)

    if stage_configs is None:
        stage_configs = set_stage_configs(shots, epochs, use_stage_configs)


    final_model   = None
    final_trainer = None
    best_val_acc  = -np.inf
    best_ckpt     = None

    # common metric & datasets
    acc_fn     = lambda y_hat, y: np.mean(np.argmax(y_hat,1) == np.argmax(y,1))
    eval_funcs = {'accuracy': acc_fn}
    val_ds     = Dataset(val_circuits, val_labels, shuffle=False)
    test_ds    = Dataset(test_circuits, test_labels, shuffle=False)


    for idx, cfg in enumerate(stage_configs, start=1):
        t_epochs  = cfg['epochs']
        t_shots   = cfg['shots']
        opt_hp  = cfg['optimizer_hparams']
        batch   = cfg.get('batch_size', batch_size)
        t_seed    = cfg.get('seed', seed + idx)

        print(f"\n=== Stage {idx}: epochs={t_epochs}, shots={t_shots}, batch={batch}, seed={t_seed} ===")

        backend_config = {
            'backend':     backend,
            'compilation': comp_pass,
            'shots':       t_shots,
        }

        ckpt_path = os.path.join(checkpoint_dir, f'model_stage{idx}.lt')

        if final_model is None:
            # First stage: build from scratch
            diagrams = train_circuits + val_circuits
            model = TketModel.from_diagrams(diagrams, backend_config=backend_config)
            model.initialise_weights()
        else:
            # Subsequent stages: resume from last checkpoint
            model = TketModel.from_checkpoint(best_ckpt, backend_config=backend_config)


        trainer = QuantumTrainer(
            model,
            loss_function      = BinaryCrossEntropyLoss(),
            optimizer          = SPSAOptimizer,
            optim_hyperparams  = opt_hp,
            evaluate_functions = eval_funcs,
            evaluate_on_train  = True,
            epochs             = t_epochs,
            seed               = t_seed,
            verbose            = 'text'
        )


        # Fit with early stopping
        train_ds = Dataset(train_circuits, train_labels, batch_size=batch, shuffle=True)
        hist = trainer.fit(
            train_ds,
            val_ds,
            early_stopping_criterion = 'accuracy',
            early_stopping_interval  = 3,
            minimize_criterion       = False
        )

        # Save checkpoint
        model.save(ckpt_path)
        print(f"» Saved checkpoint: {ckpt_path}")

        # 5) Track best val accuracy
        #    hist.val_metrics is a list of dicts of per-epoch val metrics
        #    (you may need to inspect trainer.history if API differs)
        final_val_acc = 0.1
        # final_val_acc = hist.val_metrics[-1]['accuracy']
        if final_val_acc > best_val_acc:
            best_val_acc = final_val_acc
            best_ckpt    = ckpt_path

        # Prepare for next stage
        final_model   = model
        final_trainer = trainer

    test_ds = Dataset(test_circuits, test_labels, shuffle=False)
    test_metrics = final_trainer.evaluate(test_ds)
    print(f"\nFinal test metrics: {test_metrics}")

    return final_model, final_trainer, test_metrics

In [ ]:
# Debugging parameters:
model, trainer, metrics = train_quantum_summarizer(
  encoded_train   = ds_test,
  encoded_val     = ds_val,
  encoded_test    = ds_test,

  batch_size = 4,
  epochs     = 1,
  shots      = 32768,
  seed       = 42,
  checkpoint_dir = 'saves/debugging_model_checkpoints',
  use_stage_configs = False
)

# batch_size = 5, epochs = 1, shots = 32, seed = 42,
# train/time: 2m36s   train/time_per_epoch: 2m36s   train/time_per_step: 2.95s   valid/time: 1m16s   valid/time_per_eval: 1m16s

# epochs=1, shots=4, batch=5, seed=42
# train/time: 2m33s   train/time_per_epoch: 2m33s   train/time_per_step: 2.88s   valid/time: 1m15s   valid/time_per_eval: 1m15s

# epochs=1, shots=4, batch=64, seed=42
# train/time: 2m30s   train/time_per_epoch: 2m30s   train/time_per_step: 29.99s   valid/time: 1m14s   valid/time_per_eval: 1m14s

# epochs=1, shots=1, batch=128, seed=42
# train/time: 2m31s   train/time_per_epoch: 2m31s   train/time_per_step: 50.34s   valid/time: 1m14s   valid/time_per_eval: 1m14s



In [ ]:
# Custom 2-stage strategy:
stages = [
  {'epochs': 5,  'shots': 1024,
   'optimizer_hparams': {'a':0.15,'c':0.06,'A':0.2*5},
   'batch_size':  8},
  {'epochs': 15, 'shots': 8192,
   'optimizer_hparams': {'a':0.02,'c':0.02,'A':0.2*15}}
]

model, trainer, metrics = train_quantum_summarizer(
  encoded_train   = ds_test,
  encoded_val     = ds_val,
  encoded_test    = ds_test,

  batch_size = 5,
  seed       = 123,
  # epochs = 10,
  # shots = 8192,
  stage_configs   = stages
)

